In [31]:
import glob
from collections import deque, defaultdict
import networkx as nx
import tqdm

In [32]:
ANORMAL_DATASET_FILES = glob.glob("data/sendmail/anom/**/*.int.gz")
NORMAL_DATASET_FILES = glob.glob("data/sendmail/normal/*.int.gz")
MAP_FILE = "data/sendmail/map.int"
print(f"Number of anomalous dataset files: {len(ANORMAL_DATASET_FILES)}")
print(f"Number of normal dataset files: {len(NORMAL_DATASET_FILES)}")

Number of anomalous dataset files: 16
Number of normal dataset files: 7


In [33]:
int_map = {}
with open(MAP_FILE, "r") as f:
    for i, line in enumerate(f):
        syscall = line.strip()
        int_map[i+1] = syscall

In [34]:
def get_syscall_sequence(file_path):
    with open(file_path, 'rt') as f:
        return [int(line.strip().split(' ')[1]) for line in f]

In [35]:
# def k_map_from_graph(graph, pagerank_scores):
#     k_map = {}
#     for edge in graph.edges(data=True):
#         src, dst, data = edge
#         k = 1 - (pagerank_scores[src] * data['weight'] / graph.out_degree(src, weight='weight') / pagerank_scores[dst])
#         k_map[(src, dst)] = k
#     return k_map
def k_map_from_graph(graph, pagerank_scores):
    k_map = {}
    for edge in graph.edges(data=True):
        src, dst, data = edge
        
        # Calculate the raw contribution fraction
        out_degree = graph.out_degree(src, weight='weight')
        if out_degree == 0 or pagerank_scores[dst] == 0:
            k = 1.0 # Max suspicion if we can't calculate a valid transition
        else:
            contribution = (pagerank_scores[src] * data['weight'] / out_degree) / pagerank_scores[dst]
            k = 1 - contribution
        
        # Clip to [0, 1] to correct for Damping Factor inflation/floating point errors
        k_map[(src, dst)] = max(0.0, min(1.0, k)) 
    return k_map

def distance_to_pattern(observed_sequence, pattern_sequence, weight_map, prev_char=None):
    assert len(observed_sequence) == len(pattern_sequence), "Sequences must be of the same length"
    distance = 0.0
    for i in range(len(pattern_sequence)):
        if i >= len(observed_sequence):
            break
        if observed_sequence[i] != pattern_sequence[i]:
            if i == 0:
                distance += weight_map.get((prev_char, observed_sequence[i]), 0.1)
            else:
                distance += weight_map.get((observed_sequence[i - 1], observed_sequence[i]), 0.1)
    return distance


In [36]:
def graph_from_sequence(sequence):
    graph = nx.DiGraph()
    for i in range(len(sequence) - 1):
        src = sequence[i]
        dst = sequence[i + 1]
        if graph.has_edge(src, dst):
            graph[src][dst]['weight'] += 1
        else:
            graph.add_edge(src, dst, weight=1)
    return graph

def sequences_from_sequence(sequence, window_size=6):
    sequences = []
    current_sequence = deque(maxlen=window_size)
    for syscall in sequence:
        current_sequence.append(syscall)
        if len(current_sequence) == window_size:
            sequences.append(list(current_sequence))
    return sequences

In [37]:
class IntrusionDetector:
    def _init_pattern_library(self):
        normal_sequence = get_syscall_sequence(self.normal_file)
        self.normal_graph = graph_from_sequence(normal_sequence)
        assert self.normal_graph.number_of_nodes() == len(set(normal_sequence)), "Graph nodes should match unique syscalls in the sequence"
        
        self.pagerank_scores = nx.pagerank(self.normal_graph, weight='weight')
        self.normal_sequences = sequences_from_sequence(normal_sequence, window_size=self.window_size)
        self.weight_map = k_map_from_graph(self.normal_graph, self.pagerank_scores)
        assert len(self.weight_map) == self.normal_graph.number_of_edges(), "Weight map should have the same number of entries as graph edges"

    def __init__(self, file, window_size=6, h_distance_threshold=0.5, anomality_rate_threshold=0.5):
        self.normal_file = file
        self.window_size = window_size
        self.h_distance_threshold = h_distance_threshold
        self.anomality_rate_threshold = anomality_rate_threshold

    def _evaluate_sequence(self, observed_sequence, prev_char=None):
        min_distance = float('inf')
        # print(f"Evaluating observed sequence: {observed_sequence}")
        for pattern_sequence in self.normal_sequences:
            if observed_sequence == pattern_sequence:
                # print("Exact match found")
                distance = 0.0
            else:
                distance = distance_to_pattern(
                    observed_sequence,
                    pattern_sequence,
                    self.weight_map,
                    prev_char=prev_char,
                )
                if distance < 0.0:
                    print(f"Negative distance calculated: {distance} for observed sequence {observed_sequence} and pattern {pattern_sequence}")
                # print(f"Distance to pattern “{pattern_sequence}”: {distance}")
            min_distance = min(min_distance, distance)
            if min_distance == 0.0:
                break
        return min_distance
    
    def fit(self):
        self._init_pattern_library()

    def predict(self, sequence):
        observed_sequences = sequences_from_sequence(sequence, window_size=self.window_size)
        distances = []
        for i, obs_seq in enumerate(tqdm.tqdm(observed_sequences)):
            prev_char = sequence[i - 1] if i > 0 else None
            distance = self._evaluate_sequence(obs_seq, prev_char=prev_char)
            distances.append(distance)
        anomalities = [d > self.h_distance_threshold for d in distances]
        anomality_rate = sum(anomalities) / len(anomalities)
        return anomality_rate > self.anomality_rate_threshold, anomality_rate, distances

In [38]:
model = IntrusionDetector(NORMAL_DATASET_FILES[1], window_size=6, h_distance_threshold=0.5, anomality_rate_threshold=0.2)
model.fit()

In [41]:
# Test on an abnormal sequence
for abnormal_file in ANORMAL_DATASET_FILES:
    abnormal_sequence = get_syscall_sequence(abnormal_file)[:1000]  # Limit to first 1000 syscalls for testing
    print(f"Testing on abnormal file: {abnormal_file}, sequence length: {len(abnormal_sequence)}")
    is_anomalous, anomality_rate, distances = model.predict(abnormal_sequence)
    print(f"Is anomalous: {is_anomalous}, Anomality rate: {anomality_rate}")

Testing on abnormal file: data/sendmail/anom/syslog/syslog-local-1.int.gz, sequence length: 1000


100%|██████████| 995/995 [00:12<00:00, 80.82it/s] 


Is anomalous: False, Anomality rate: 0.13869346733668342
Testing on abnormal file: data/sendmail/anom/syslog/syslog-remote-2.int.gz, sequence length: 1000


100%|██████████| 995/995 [00:18<00:00, 53.35it/s] 


Is anomalous: True, Anomality rate: 0.264321608040201
Testing on abnormal file: data/sendmail/anom/syslog/syslog-local-2.int.gz, sequence length: 1000


...

Is anomalous: True, Anomality rate: 0.21481481481481482
Testing on abnormal file: data/sendmail/anom/fail/cert-sm5x-1.int.gz, sequence length: 1000


100%|██████████| 995/995 [00:20<00:00, 49.12it/s] 


Is anomalous: True, Anomality rate: 0.24623115577889448
Testing on abnormal file: data/sendmail/anom/decode/sm-280.int.gz, sequence length: 1000


 34%|███▎      | 334/995 [00:04<00:08, 80.04it/s] 


KeyboardInterrupt: 

In [40]:
# Test on all normal files
for normal_file in NORMAL_DATASET_FILES:
    normal_sequence = get_syscall_sequence(normal_file)[:1000]  # Limit to first 1000 syscalls for testing
    print(f"Testing on normal file: {normal_file}, sequence length: {len(normal_sequence)}")
    is_anomalous, anomality_rate, distances = model.predict(normal_sequence)
    print(f"Is anomalous: {is_anomalous}, Anomality rate: {anomality_rate}")


Testing on normal file: data/sendmail/normal/plus.int.gz, sequence length: 1000


100%|██████████| 995/995 [00:14<00:00, 68.07it/s] 


Is anomalous: False, Anomality rate: 0.16683417085427135
Testing on normal file: data/sendmail/normal/sendmail.log.int.gz, sequence length: 1000


100%|██████████| 995/995 [00:00<00:00, 3535.49it/s]


Is anomalous: False, Anomality rate: 0.0
Testing on normal file: data/sendmail/normal/bounce-1.int.gz, sequence length: 293


...

Is anomalous: True, Anomality rate: 0.20833333333333334
Testing on normal file: data/sendmail/normal/bounce.int.gz, sequence length: 818


100%|██████████| 813/813 [00:12<00:00, 64.82it/s] 


Is anomalous: False, Anomality rate: 0.17466174661746617
Testing on normal file: data/sendmail/normal/sendmail.daemon.int.gz, sequence length: 1000


 24%|██▍       | 238/995 [00:02<00:08, 89.92it/s]  


KeyboardInterrupt: 

In [ ]:
abnormal_sequence = get_syscall_sequence(ANORMAL_DATASET_FILES[0])[:1000]  # Limit to first 1000 syscalls for testing
print(f"Abnormal sequence length: {len(abnormal_sequence)}")
is_anomalous = model.predict(abnormal_sequence)
print(is_anomalous)

Abnormal sequence length: 1000


100%|██████████| 995/995 [00:16<00:00, 61.21it/s] 

(False, 0.24723618090452262, [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.7116438378488091, 1.5860921973918791, 0.7134502923976609, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

In [ ]:
def show_graph(graph, true_labels=False):
    if true_labels:
        labels = {node: int_map[node] for node in graph.nodes()}
    else:
        labels = {node: node for node in graph.nodes()}
    nx.draw(graph, with_labels=True, labels=labels)

In [ ]:
sorted_pagerank = sorted(model.pagerank_scores.items(), key=lambda x: x[1], reverse=True)
sorted_pagerank

[(104, 0.1635834986733266),
 (105, 0.12336999407603076),
 (5, 0.09490963095273539),
 (106, 0.07459408567281317),
 (19, 0.06717394743043023),
 (112, 0.06590201425827703),
 (78, 0.04179826609758538),
 (4, 0.0330730379679502),
 (50, 0.02489873012957831),
 (2, 0.02224559564332449),
 (108, 0.020651100955500545),
 (27, 0.015340613818600468),
 (3, 0.0144383859800401),
 (93, 0.012896444120386524),
 (128, 0.011954228840764396),
 (121, 0.01174134733687944),
 (1, 0.011359209324491061),
 (89, 0.010890009917424903),
 (66, 0.010430169055217087),
 (100, 0.009720250344975229),
 (122, 0.008839826664564017),
 (94, 0.007824483270829245),
 (75, 0.007670861252405767),
 (9, 0.007628783155346108),
 (17, 0.006341680587781201),
 (14, 0.0058567068957007164),
 (40, 0.005845185942406853),
 (155, 0.005668699800985709),
 (6, 0.005574027620668982),
 (123, 0.005334850659313628),
 (101, 0.005286405097183712),
 (83, 0.005239403636125316),
 (45, 0.00515238782017253),
 (32, 0.00499612813723788),
 (23, 0.00496711577069796